# Task 3: Grounded Generation with Live LLM & Schema Validation

This notebook implements **Day 3** of the AI Clinical Decision Support system:
1. **Evidence Retrieval with Citations**: Retrieves relevant chunks from the Chroma vector database.
2. **Grounded Prompt Construction**: Builds citation-bound prompts strictly constrained to retrieved evidence.
3. **Live LLM Generation**: Triggers real-time LLM generation via NVIDIA NIM API (`meta/llama-3.1-8b-instruct`) using `NV_API_KEY`.
4. **JSON Schema Enforcement**: Validates that all responses strictly conform to `schema/response_schema.json`.
5. **Abstention & Refusal Guard**: Ensures out-of-scope or unsupported queries safely abstain instead of hallucinating.

In [1]:
# Setup: Environment configuration and project file checks
import os
import sys
import json

proj_root = os.path.abspath('.')
if proj_root not in sys.path:
    sys.path.insert(0, proj_root)

# Set NVIDIA API Key
NV_API_KEY = os.environ.get('NV_API_KEY') or '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'
os.environ['NV_API_KEY'] = NV_API_KEY

print('Project root:', proj_root)
print('Python version:', sys.version.splitlines()[0])
print('NV_API_KEY configured:', NV_API_KEY[:10] + '...' + NV_API_KEY[-6:])
print()
print('Checking required files:')
for p in ['schema/response_schema.json', 'prompt/grounding_prompt.txt', 'generation.py', 'retrieval.py', 'query.py']:
    print(f" - {p}:", os.path.exists(os.path.join(proj_root, p)))


Project root: C:\Users\Maka\Downloads\ai-clinical-decision-support-day1-day-2-main\ai-clinical-decision-support-day1-day-2-main
Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
NV_API_KEY configured: nvapi-MQxY...ySLWQY

Checking required files:
 - schema/response_schema.json: True
 - prompt/grounding_prompt.txt: True
 - generation.py: True
 - retrieval.py: True
 - query.py: True


In [2]:
# Load Chroma Vector Database Index and verify retrieval
from query import load_index
from retrieval import retrieve_with_citation, print_retrieval_view

print('Loading vector index...')
vectordb = load_index()
print('Index loaded successfully.\n')

test_q = 'What is the target blood pressure for patients with cardiovascular disease?'
results = retrieve_with_citation(vectordb, test_q, k=3)
print_retrieval_view(test_q, results)


Loading vector index...
Index loaded successfully.


 CLINICAL QUERY: What is the target blood pressure for patients with cardiovascular disease?

[1] ⭐ [CONFIDENT] | Score: 0.805 | Source: Guideline for the pharmacological treatment of hypertension in adults.pdf (p. 28)
    Chunk ID: Guideline for the pharmacological treatment of hypertension in adults.pdf_p28_c1
--------------------------------------------------------------------------------
    Content: "3.6 Target blood pressure 6. RECOMMENDATION ON TARGET BLOOD PRESSURES WHO recommends a target blood pressure treatment goal of <140/90 mmHg in all patients  with hypertension without comorbidities. Strong recommendation, moderate-certainty evidence WHO recommends a target syst..."

[2] ⭐ [CONFIDENT] | Score: 0.754 | Source: WHO-NMH-NVI-18.2-eng.pdf (p. 14)
    Chunk ID: WHO-NMH-NVI-18.2-eng.pdf_p14_c1
--------------------------------------------------------------------------------
    Content: "Treatment targets For most patients, b

In [3]:
# Load Grounding Prompt Template and JSON Response Schema
prompt_path = os.path.join(proj_root, 'prompt', 'grounding_prompt.txt')
schema_path = os.path.join(proj_root, 'schema', 'response_schema.json')

with open(prompt_path, 'r', encoding='utf-8') as f:
    grounding_prompt = f.read()

with open(schema_path, 'r', encoding='utf-8') as f:
    schema = json.load(f)

print('Loaded grounding prompt preview:')
print('\n'.join(grounding_prompt.splitlines()[:6]))
print()
print('Schema keys:', list(schema.keys()))
print('Schema required properties:', schema.get('required', []))


Loaded grounding prompt preview:
You are a clinical answer assistant strictly bound to the provided retrieved evidence.

Constraints:
1. ROLE: Act only as a citation-bound assistant. Do NOT introduce facts not present in evidence.
2. CONTEXT BOUNDARY: Use only the retrieved chunks listed after the instruction. Do not access external knowledge.
3. OUTPUT FORMAT: Return a single JSON object that conforms to the project schema. Fields: status, answer, evidence (list of supporting snippets), citations (list of {document_name,page_number,chunk_id}), confidence.

Schema keys: ['$schema', 'title', 'type', 'required', 'properties', 'additionalProperties']
Schema required properties: ['status', 'answer', 'evidence', 'citations', 'confidence']


In [4]:
# Setup JSON Schema Validator
import jsonschema

validator = jsonschema.Draft7Validator(schema)

def validate_response(obj):
    errors = sorted(validator.iter_errors(obj), key=lambda e: str(e.path))
    if errors:
        print('Validation errors:')
        for err in errors:
            print(f" - Field '{'/'.join(map(str, err.path))}': {err.message}")
        return False
    return True

print('jsonschema Draft7Validator initialized and ready.')


jsonschema Draft7Validator initialized and ready.


In [5]:
# Define Live LLM generation helpers, Prompt Builder, and JSON parser
import requests
from generation import answer_question

def build_prompt(retrieved, question):
    context_blocks = []
    for r in retrieved:
        context_blocks.append(
            f"SOURCE: {r['document_name']} (page {r['page_number']}, chunk_id: {r.get('chunk_id')})\n{r['text'][:800]}"
        )
    prompt = grounding_prompt.replace('{context_blocks}', '\n\n'.join(context_blocks))
    prompt = prompt.replace('{question}', question)
    return prompt

def call_live_llm(prompt_text, nv_api_key=None, endpoint=None, model=None, timeout=30):
    endpoint = endpoint or os.environ.get('NV_LLM_ENDPOINT') or 'https://integrate.api.nvidia.com/v1/chat/completions'
    nv_api_key = nv_api_key or os.environ.get('NV_API_KEY')
    model = model or os.environ.get('NV_MODEL') or 'meta/llama-3.1-8b-instruct'

    if not nv_api_key:
        raise RuntimeError('NV_API_KEY not set in environment for live mode')

    headers = {
        'Authorization': f'Bearer {nv_api_key}',
        'Content-Type': 'application/json',
        'Accept': 'application/json'
    }

    system_message = (
        "You are an AI clinical decision support assistant. Return ONLY a single valid JSON object strictly conforming to this schema:\n"
        "- 'status': string, MUST be either 'grounded' or 'abstain'\n"
        "- 'answer': string (the concise recommendation if grounded, or empty string '' if abstain)\n"
        "- 'evidence': array of string snippets from the provided text supporting the answer\n"
        "- 'citations': array of objects, each with 'document_name' (string) and 'page_number' (integer or string), and optional 'chunk_id' (string)\n"
        "- 'confidence': string, MUST be either 'confident', 'uncertain', or 'insufficient'\n"
        "Do NOT include markdown formatting or extra keys."
    )

    payload = {
        'model': model,
        'messages': [
            {'role': 'system', 'content': system_message},
            {'role': 'user', 'content': prompt_text}
        ],
        'temperature': 0.1,
        'max_tokens': 1024
    }

    r = requests.post(endpoint, headers=headers, json=payload, timeout=timeout)
    r.raise_for_status()
    data = r.json()

    if isinstance(data, dict) and 'choices' in data and len(data['choices']) > 0:
        c = data['choices'][0]
        if isinstance(c, dict) and 'message' in c and 'content' in c['message']:
            return c['message']['content']
        elif isinstance(c, dict) and 'text' in c:
            return c['text']
    return json.dumps(data)

def parse_and_clean_json(raw_text, top_score=0.0):
    text = raw_text.strip()
    if text.startswith('```'):
        lines = text.splitlines()
        if lines[0].startswith('```'):
            lines = lines[1:]
        if lines and lines[-1].startswith('```'):
            lines = lines[:-1]
        text = '\n'.join(lines).strip()

    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end != -1:
        text = text[start:end+1]

    obj = json.loads(text)

    # Normalize status
    status_raw = str(obj.get('status', '')).lower()
    if status_raw in ('grounded', 'answer', 'success', 'ok', 'answered'):
        status = 'grounded'
    elif status_raw in ('abstain', 'refusal', 'refuse', 'rejected', 'insufficient'):
        status = 'abstain'
    else:
        status = 'grounded' if obj.get('answer') else 'abstain'

    # Normalize confidence
    confidence_raw = str(obj.get('confidence', '')).lower()
    if confidence_raw in ('confident', 'high', 'strong'):
        confidence = 'confident'
    elif confidence_raw in ('uncertain', 'medium', 'moderate'):
        confidence = 'uncertain'
    elif confidence_raw in ('insufficient', 'low', 'none', 'unknown'):
        confidence = 'insufficient'
    else:
        confidence = 'confident' if status == 'grounded' else 'insufficient'

    # Clean citations
    valid_citations = []
    for cit in obj.get('citations', []):
        if isinstance(cit, dict) and 'document_name' in cit and 'page_number' in cit:
            c_dict = {
                'document_name': str(cit['document_name']),
                'page_number': int(cit['page_number']) if str(cit['page_number']).isdigit() else str(cit['page_number'])
            }
            if 'chunk_id' in cit and cit['chunk_id']:
                c_dict['chunk_id'] = str(cit['chunk_id'])
            valid_citations.append(c_dict)

    # Clean evidence
    evidence_list = []
    for ev in obj.get('evidence', []):
        if isinstance(ev, str) and ev.strip():
            evidence_list.append(ev.strip())

    cleaned = {
        'status': status,
        'answer': str(obj.get('answer', '')),
        'evidence': evidence_list,
        'citations': valid_citations,
        'confidence': confidence,
    }
    if top_score is not None:
        cleaned['top_score'] = round(float(top_score), 3)

    return cleaned

def simulate_llm_response(question, retrieved):
    resp = answer_question(vectordb, question, k=3)
    evidence = []
    for ch in resp.get('grounded_chunks', []):
        txt = ch.get('text') or ch.get('page_content') or ch.get('text_excerpt', '')
        if txt:
            evidence.append(txt.strip()[:400])
    out = {
        'status': resp.get('status', 'abstain'),
        'answer': resp.get('answer', ''),
        'evidence': evidence,
        'citations': resp.get('citations', []),
        'confidence': resp.get('confidence', 'uncertain'),
        'top_score': resp.get('top_score', 0.0)
    }
    return out

def generate_response(question, mode='live'):
    retrieved = retrieve_with_citation(vectordb, question, k=3)
    top_score = retrieved[0]['score'] if retrieved else 0.0
    prompt_text = build_prompt(retrieved, question)
    
    if mode == 'live':
        try:
            print(f"Triggering Live LLM generation via NVIDIA NIM API...")
            raw = call_live_llm(prompt_text)
            candidate = parse_and_clean_json(raw, top_score=top_score)
        except Exception as e:
            print('Live LLM call failed, falling back to simulation:', e)
            candidate = simulate_llm_response(question, retrieved)
    else:
        candidate = simulate_llm_response(question, retrieved)
        
    ok = validate_response(candidate)
    return candidate, ok

print('generate_response pipeline initialized successfully.')


generate_response pipeline initialized successfully.


In [6]:
# Execute Live LLM Generation Tests: In-Scope, Out-of-Scope, and Paraphrased Questions
in_scope = 'What is the target blood pressure for patients with cardiovascular disease?'
out_scope = 'What is the recommended treatment for spotted fever in alpacas?'
paraphrase = 'What blood pressure target should be used for someone with heart disease?'

print('======================================================================')
print('TEST 1: IN-SCOPE CLINICAL QUERY (mode=live)')
print(f'Query: {in_scope}')
print('======================================================================')
resp_in, ok_in = generate_response(in_scope, mode='live')
print('\nIN-SCOPE RESPONSE:')
print(json.dumps(resp_in, indent=2))
print(f'Schema valid: {ok_in}')

print('\n======================================================================')
print('TEST 2: OUT-OF-SCOPE QUERY (mode=live)')
print(f'Query: {out_scope}')
print('======================================================================')
resp_out, ok_out = generate_response(out_scope, mode='live')
print('\nOUT-OF-SCOPE RESPONSE:')
print(json.dumps(resp_out, indent=2))
print(f'Schema valid: {ok_out}')

print('\n======================================================================')
print('TEST 3: PARAPHRASED QUERY (mode=live)')
print(f'Query: {paraphrase}')
print('======================================================================')
resp_para, ok_para = generate_response(paraphrase, mode='live')
print('\nPARAPHRASED RESPONSE:')
print(json.dumps(resp_para, indent=2))
print(f'Schema valid: {ok_para}')

# Assertions
assert ok_in and resp_in['status'] == 'grounded' and len(resp_in['citations']) > 0, "In-scope validation failed"
assert ok_out and resp_out['status'] == 'abstain', "Out-of-scope validation failed"
assert ok_para and resp_para['status'] == 'grounded', "Paraphrase validation failed"
print('\n======================================================================')
print('ALL LIVE LLM GENERATION & SCHEMA VALIDATION TESTS COMPLETED SUCCESSFULLY!')
print('======================================================================')


TEST 1: IN-SCOPE CLINICAL QUERY (mode=live)
Query: What is the target blood pressure for patients with cardiovascular disease?
Triggering Live LLM generation via NVIDIA NIM API...

IN-SCOPE RESPONSE:
{
  "status": "grounded",
  "answer": "A target systolic blood pressure treatment goal of <130 mmHg is recommended for patients with hypertension and known cardiovascular disease.",
  "evidence": [
    "WHO recommends a target systolic blood pressure treatment goal of <130 mmHg in patients with hypertension and known cardiovascular disease (CVD).",
    "WHO recommends a target systolic blood pressure treatment goal of <130 mmHg in patients with hypertension and known cardiovascular disease (CVD)."
  ],
  "citations": [
    {
      "document_name": "Guideline for the pharmacological treatment of hypertension in adults.pdf",
      "page_number": 28,
      "chunk_id": "Guideline for the pharmacological treatment of hypertension in adults.pdf_p28_c1"
    },
    {
      "document_name": "Guidel